<a href="https://colab.research.google.com/github/adelinazzh/Tugas_PSD_M5/blob/main/Tugas_PSD_M5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Membaca dataset
import pandas as pd
import numpy as np
from IPython.display import display

In [2]:
df = pd.read_csv("paket_a_transaksi_pos_2024_2025.csv")

print("Dataset berhasil dibaca!")
print("Ukuran dataset:", df.shape)

display(df.head())

Dataset berhasil dibaca!
Ukuran dataset: (5030, 9)


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500,0,SELESAI
4,TRX0243509,26/03/2025,G05,K016,RMT1407,1,6000,0,SELESAI


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5030 entries, 0 to 5029
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id_transaksi   5030 non-null   object
 1   tanggal        5030 non-null   object
 2   kode_gerai     5030 non-null   object
 3   kode_kasir     4769 non-null   object
 4   kode_produk    5030 non-null   object
 5   jumlah         5030 non-null   int64 
 6   harga_satuan   5030 non-null   object
 7   diskon_persen  5030 non-null   int64 
 8   status         5030 non-null   object
dtypes: int64(2), object(7)
memory usage: 353.8+ KB


df.info() ini untuk menampilkan nama kolom, jumlah data yang terisi, tipe data, dan penggunaan memori.

In [4]:
display(df.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id_transaksi,5030,5000,TRX0244203,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tanggal,5030,902,2025-06-01,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN
kode_gerai,5030,12,G01,478,NaN,NaN,NaN,NaN,NaN,NaN,NaN
kode_kasir,4769,40,K021,142,NaN,NaN,NaN,NaN,NaN,NaN,NaN
kode_produk,5030,216,SGR1204,96,NaN,NaN,NaN,NaN,NaN,NaN,NaN
jumlah,5030.0,NaN,NaN,NaN,5.588469,68.788887,-12.0,1.0,2.0,3.0,2375.0
harga_satuan,5030,147,11000,197,NaN,NaN,NaN,NaN,NaN,NaN,NaN
diskon_persen,5030.0,NaN,NaN,NaN,2.563618,5.211135,0.0,0.0,0.0,0.0,20.0
status,5030,3,SELESAI,4749,NaN,NaN,NaN,NaN,NaN,NaN,NaN


di kode ini, saya bisa melihat gambaran umum dari data yang kita punya, mulai dari jumlah data, nilai terkecil dan terbesar, sampai kategori yang ada di dalam dataset

In [5]:
#Missing value
missing = pd.DataFrame({
    "Jumlah Missing": df.isna().sum(),
    "Persentase Missing (%)": (
        df.isna().sum() / len(df) * 100
    ).round(2)
})

display(missing[missing["Jumlah Missing"] > 0])

,Jumlah Missing,Persentase Missing (%)
kode_kasir,261,5.19


di bagian ini, saya mengecek apakah ada data yang kosong di dalam dataset. dari hasil pengecekan, ternyata ada nilai kosong di kolom kode_kasir

In [6]:
df["kode_kasir"] = df["kode_kasir"].fillna("TIDAK DIKETAHUI")

print("Missing value setelah penanganan:")
display(df.isna().sum().to_frame("Jumlah Missing"))

Missing value setelah penanganan:


,Jumlah Missing
id_transaksi,0
tanggal,0
kode_gerai,0
kode_kasir,0
kode_produk,0
jumlah,0
harga_satuan,0
diskon_persen,0
status,0


di bagian ini, data yang kosong pada kolom kode_kasir saya ganti dengan tulisan TIDAK DIKETAHUI, karena kolom ini berisi kode kasir, jadi kurang cocok kalau diisi dengan rata-rata. dengan begitu, data transaksi lainnya tetap bisa digunakan untuk proses analisis selama informasi yang dibutuhkan masih lengkap.

In [7]:
#membersihkan data harga, tanggal, dan diskon

#mengubah harga_satuan menjadi numerik
harga = (
    df["harga_satuan"]
    .astype("string")
    .str.strip()
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

df["harga_satuan"] = pd.to_numeric(harga, errors="coerce")

print("Tipe data harga_satuan:", df["harga_satuan"].dtype)
display(df[["harga_satuan"]].head(10))

Tipe data harga_satuan: Float64


,harga_satuan
0,13000.0
1,21000.0
2,7500.0
3,6500.0
4,6000.0
5,45500.0
6,18000.0
7,47500.0
8,11000.0
9,6500.0


di bagian ini, saya mengubah format harga supaya semuanya menjadi angka yang seragam dan bisa digunakan untuk perhitungan. contohnya, harga 13.000 diubah menjadi 13000, sedangkan 7500,00 diubah menjadi 7500.00. jadi, nantinya data harga bisa dihitung dengan benar saat saya mengolah total penjualan dan pendapatan bersih.

In [9]:
#mengubah kolom tanggal menjadi datetime
df["tanggal"] = pd.to_datetime(
    df["tanggal"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

print("Tipe data tanggal:", df["tanggal"].dtype)
display(df[["tanggal"]].head(10))

print("Tanggal yang gagal dikonversi:", df["tanggal"].isna().sum())

Tipe data tanggal: datetime64[ns]


,tanggal
0,2025-06-02
1,2024-12-28
2,2024-09-04
3,2025-07-30
4,2025-03-26
5,2025-07-18
6,2025-09-24
7,2024-10-09
8,2024-03-20
9,2024-03-22


Tanggal yang gagal dikonversi: 0


di bagian ini, saya mengubah kolom tanggal menjadi format datetime supaya nantinya bisa diolah dan diambil tahunnya. karena format tanggal di dataset ada yang berbeda, saya menggunakan format="mixed" agar bisa membaca berbagai format tanggal. kalau ada tanggal yang tidak bisa dibaca, nilainya akan menjadi NaT dan tidak akan digunakan dalam perhitungan pendapatan tahunan

In [10]:
df["diskon_persen"] = pd.to_numeric(
    df["diskon_persen"],
    errors="coerce"
)

print("Nilai diskon yang tersedia:")
display(
    df["diskon_persen"]
    .value_counts(dropna=False)
    .sort_index()
)

Nilai diskon yang tersedia:


,count
diskon_persen,
0,3848
5,346
10,420
15,271
20,145


di bagian ini, saya memastikan kolom diskon_persen sudah berbentuk angka supaya bisa digunakan dalam perhitungan. misalnya, nilai 10 berarti diskon sebesar 10%, bukan potongan harga sebesar rp10. jadi, nantinya diskon bisa dihitung dengan benar saat saya mencari pendapatan bersih dari setiap transaksi

In [11]:
df["status"] = df["status"].astype("string").str.strip().str.upper()

print("Jumlah transaksi berdasarkan status:")
display(
    df["status"]
    .value_counts(dropna=False)
    .rename_axis("Status")
    .to_frame("Jumlah Transaksi")
)

Jumlah transaksi berdasarkan status:


,Jumlah Transaksi
Status,
SELESAI,4749
BATAL,191
RETUR,90


di bagian ini, untuk memeriksa status dari setiap data transaksi

In [12]:
#menghitung pendapatan bersih setelah diskon

#total harga sebelum diskon
df["total_sebelum_diskon"] = (
    df["jumlah"] * df["harga_satuan"]
)

#nilai potongan diskon
df["nilai_diskon"] = (
    df["total_sebelum_diskon"]
    * df["diskon_persen"] / 100
)

#pendapatan bersih setelah diskon
df["pendapatan_bersih"] = (
    df["total_sebelum_diskon"] - df["nilai_diskon"]
)

display(
    df[
        [
            "id_transaksi",
            "jumlah",
            "harga_satuan",
            "diskon_persen",
            "total_sebelum_diskon",
            "nilai_diskon",
            "pendapatan_bersih",
            "status"
        ]
    ].head(10)
)

,id_transaksi,jumlah,harga_satuan,diskon_persen,total_sebelum_diskon,nilai_diskon,pendapatan_bersih,status
0,TRX0241065,4,13000.0,0,52000.0,0.0,52000.0,SELESAI
1,TRX0240510,1,21000.0,0,21000.0,0.0,21000.0,SELESAI
2,TRX0244483,3,7500.0,0,22500.0,0.0,22500.0,SELESAI
3,TRX0242532,2,6500.0,0,13000.0,0.0,13000.0,SELESAI
4,TRX0243509,1,6000.0,0,6000.0,0.0,6000.0,SELESAI
5,TRX0240658,12,45500.0,10,546000.0,54600.0,491400.0,SELESAI
6,TRX0244581,1,18000.0,0,18000.0,0.0,18000.0,SELESAI
7,TRX0241251,4,47500.0,10,190000.0,19000.0,171000.0,SELESAI
8,TRX0244274,3,11000.0,0,33000.0,0.0,33000.0,SELESAI
9,TRX0243163,2,6500.0,0,13000.0,0.0,13000.0,BATAL


di bagian ini, saya mulai menghitung total harga dari setiap transaksi dengan mengalikan jumlah barang dan harga satuannya. setelah itu, saya menghitung potongan harga berdasarkan persentase diskon yang diberikan. jadi, pendapatan bersih yang diperoleh sudah dikurangi diskon. contohnya, jika membeli 4 barang dengan harga rp13.000 per barang dan mendapatkan diskon 10%, maka total awalnya rp52.000 dan setelah dikurangi diskon menjadi rp46.800

In [13]:
#status yang diperbolehkan untuk analisis
status_valid = ["SELESAI"]

#kolom yang wajib valid untuk perhitungan pendapatan
kolom_wajib = [
    "tanggal",
    "jumlah",
    "harga_satuan",
    "diskon_persen",
    "status"
]

df_valid = df[
    df["status"].isin(status_valid)
    & df[kolom_wajib].notna().all(axis=1)
    & (df["jumlah"] >= 0)
    & (df["harga_satuan"] >= 0)
    & df["diskon_persen"].between(0, 100)
].copy()

print("Total baris dataset:", len(df))
print("Transaksi valid untuk analisis:", len(df_valid))
print("Transaksi yang tidak digunakan:", len(df) - len(df_valid))

Total baris dataset: 5030
Transaksi valid untuk analisis: 4712
Transaksi yang tidak digunakan: 318


di bagian ini, saya menyaring data transaksi yang akan digunakan untuk analisis. saya hanya mengambil transaksi dengan status SELESAI dan memastikan tanggal, jumlah barang, harga, serta diskonnya memiliki nilai yang valid. sementara itu, transaksi dengan status BATAL dan RETUR tidak saya ikutkan dalam perhitungan, supaya hasil pendapatan yang diperoleh hanya berasal dari transaksi yang berhasil

In [14]:
#menghitung pendapatan tahun 2024 dan 2025
df_valid["tahun"] = df_valid["tanggal"].dt.year

display(
    df_valid[["tanggal", "tahun", "pendapatan_bersih"]].head(10)
)

,tanggal,tahun,pendapatan_bersih
0,2025-06-02,2025,52000.0
1,2024-12-28,2024,21000.0
2,2024-09-04,2024,22500.0
3,2025-07-30,2025,13000.0
4,2025-03-26,2025,6000.0
5,2025-07-18,2025,491400.0
6,2025-09-24,2025,18000.0
7,2024-10-09,2024,171000.0
8,2024-03-20,2024,33000.0
10,2024-04-19,2024,120600.0


di bagian ini, saya mengambil tahun dari setiap tanggal transaksi menggunakan .dt.year. tujuannya supaya data transaksi bisa dikelompokkan berdasarkan tahunnya, yaitu 2024 dan 2025. jadi, nantinya kita bisa menghitung dan membandingkan total pendapatan dari kedua tahun tersebut

In [15]:
#tahun yang akan dianalisis
tahun_analisis = [2024, 2025]

#menghitung total pendapatan bersih
hasil_pendapatan = (
    df_valid[df_valid["tahun"].isin(tahun_analisis)]
    .groupby("tahun")["pendapatan_bersih"]
    .sum()
    .reindex(tahun_analisis, fill_value=0)
    .reset_index()
)

#merapikan nama kolom
hasil_pendapatan.columns = [
    "Tahun",
    "Total Pendapatan Bersih (Rp)"
]

#format rupiah tanpa angka desimal
hasil_pendapatan["Total Pendapatan Bersih (Rp)"] = (
    hasil_pendapatan["Total Pendapatan Bersih (Rp)"]
    .round(0)
    .astype("int64")
)

display(hasil_pendapatan)

,Tahun,Total Pendapatan Bersih (Rp)
0,2024,387847925
1,2025,259271725


di bagian ini, saya menjumlahkan seluruh pendapatan bersih dari transaksi yang sudah valid berdasarkan masing-masing tahun. hasilnya akan ditampilkan dalam bentuk tabel yang berisi total pendapatan tahun 2024 dan 2025. dari tabel tersebut saya bisa melihat berapa pendapatan yang diperoleh pada setiap tahunnya dan membandingkan hasilnya

In [16]:
pendapatan_2024 = hasil_pendapatan.loc[
    hasil_pendapatan["Tahun"] == 2024,
    "Total Pendapatan Bersih (Rp)"
].iloc[0]

pendapatan_2025 = hasil_pendapatan.loc[
    hasil_pendapatan["Tahun"] == 2025,
    "Total Pendapatan Bersih (Rp)"
].iloc[0]

selisih = pendapatan_2025 - pendapatan_2024

if pendapatan_2024 != 0:
    persentase_perubahan = selisih / pendapatan_2024 * 100
else:
    persentase_perubahan = np.nan

print(f"Pendapatan bersih 2024: Rp{pendapatan_2024:,.0f}")
print(f"Pendapatan bersih 2025: Rp{pendapatan_2025:,.0f}")
print(f"Selisih pendapatan: Rp{selisih:,.0f}")

if pd.notna(persentase_perubahan):
    print(f"Perubahan pendapatan: {persentase_perubahan:.2f}%")
else:
    print("Persentase perubahan tidak dapat dihitung karena pendapatan 2024 nol.")

Pendapatan bersih 2024: Rp387,847,925
Pendapatan bersih 2025: Rp259,271,725
Selisih pendapatan: Rp-128,576,200
Perubahan pendapatan: -33.15%


di bagian ini, saya membandingkan total pendapatan bersih tahun 2024 dan 2025 untuk melihat apakah pendapatan mengalami kenaikan atau penurunan. saya juga menghitung selisih dan persentase perubahannya, sehingga bisa diketahui seberapa besar perubahan pendapatan dari kedua tahun tersebut.

kesimpulan

jadi, berdasarkan hasil perhitungan yang sudah dilakukan, total pendapatan bersih pada tahun 2024 adalah Rp 387.847.925, sedangkan pada tahun 2025 adalah Rp 259.271.725.

hasil ini menunjukkan bahwa pendapatan pada tahun 2025 mengalami penurunan dibandingkan tahun 2024, dengan selisih sebesar Rp 128.576.200 atau sekitar 33,15%

jadi, berdasarkan dataset yang dianalisis, pendapatan penjualan pada tahun 2024 lebih tinggi dibandingkan tahun 2025

In [17]:
#menyimpan tabel ringkasan pendapatan
hasil_pendapatan.to_csv(
    "hasil_pendapatan_2024_2025.csv",
    index=False
)

#menyimpan dataset yang telah dibersihkan
df_valid.to_csv(
    "transaksi_valid_bersih.csv",
    index=False
)

print("File hasil analisis berhasil disimpan.")

File hasil analisis berhasil disimpan.
